In [2]:
import pandas as pd

path = "Part2/Data Map-export.csv"

with open(path, "r", encoding="utf-8", errors="replace") as f:
    for i in range(10):
        print(f"{i+1:02d}:", f.readline().rstrip("\n"))


01: ﻿Template,Start Year,Region,Form of Violence,Type of Measure,Country,Title,Description
02: "Measure","1984","Africa","Sexual violence","Laws > Violence against women > Legislation","Algeria","Article 336 of the Penal Code","Article 336 of the Penal Code makes rape a punishable offence, but does not define rape.  The definition of rape and sexual crimes, as well as the criminalization of abortion, are currently being discussed by a Commission in charge of revising the Penal Code."
03: "Measure","1996","Africa","Violence against women and girls","Laws > Violence against women > Constitutional provision","Algeria","Article 34 of the Constitution","<p>Article 34 of the Constitution adopted in 1996, and amended in 2008, includes the following provisions: The State shall guarantee the inviolability of the human person. Any form of physical or moral violence or infringement of dignity shall be prohibited.</p>"
04: "Measure","1996","Africa","Violence against women and girls","Laws > Violen

In [4]:
import pandas as pd

path = "Part2/Data Map-export.csv"

df_raw = pd.read_csv(
    path,
    sep=",",
    engine="python",
    encoding="utf-8-sig",
    quotechar='"'
)

print(df_raw.shape)
print(df_raw.columns)
df_raw.head()


ParserError: ',' expected after '"'

In [6]:
df_raw = pd.read_csv(
    path,
    sep=",",
    engine="python",
    encoding="utf-8-sig",
    quotechar='"',
    on_bad_lines="skip"
)
print(df_raw.shape)
print(df_raw.columns)
df_raw.head()

(9767, 8)
Index(['Template', 'Start Year', 'Region', 'Form of Violence',
       'Type of Measure', 'Country', 'Title', 'Description'],
      dtype='object')


,Template,Start Year,Region,Form of Violence,Type of Measure,Country,Title,Description
0,Measure,1984,Africa,Sexual violence,Laws > Violence against women > Legislation,Algeria,Article 336 of the Penal Code,Article 336 of the Penal Code makes rape a pun...
1,Measure,1996,Africa,Violence against women and girls,Laws > Violence against women > Constitutional...,Algeria,Article 34 of the Constitution,<p>Article 34 of the Constitution adopted in 1...
2,Measure,1996,Africa,Violence against women and girls,Laws > Violence against women > Constitutional...,Algeria,Article 35 of the Constitution,<p>Article 35 of the Constitution adopted in 1...
3,Measure,2000,Africa,Domestic violence/Intimate partner violence,Research and statistical data > Other research...,Algeria,Lancement De La Stratégie Nationale De Lutte C...,L'enquête / étude sur la violence domestique e...
4,Measure,2001,Africa,Domestic violence/Intimate partner violence; \...,Research and statistical data > Administrative...,Algeria,Police Survey on Violence against Women,"During the third trimester of 2001, a survey u..."


In [7]:
df = df_raw.copy()

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

df.head()


,template,start_year,region,form_of_violence,type_of_measure,country,title,description
0,Measure,1984,Africa,Sexual violence,Laws > Violence against women > Legislation,Algeria,Article 336 of the Penal Code,Article 336 of the Penal Code makes rape a pun...
1,Measure,1996,Africa,Violence against women and girls,Laws > Violence against women > Constitutional...,Algeria,Article 34 of the Constitution,<p>Article 34 of the Constitution adopted in 1...
2,Measure,1996,Africa,Violence against women and girls,Laws > Violence against women > Constitutional...,Algeria,Article 35 of the Constitution,<p>Article 35 of the Constitution adopted in 1...
3,Measure,2000,Africa,Domestic violence/Intimate partner violence,Research and statistical data > Other research...,Algeria,Lancement De La Stratégie Nationale De Lutte C...,L'enquête / étude sur la violence domestique e...
4,Measure,2001,Africa,Domestic violence/Intimate partner violence; \...,Research and statistical data > Administrative...,Algeria,Police Survey on Violence against Women,"During the third trimester of 2001, a survey u..."


In [8]:
df["start_year"] = pd.to_numeric(df["start_year"], errors="coerce").astype("Int64")
df = df.dropna(subset=["start_year"])


In [9]:
df["form_of_violence"] = (
    df["form_of_violence"]
      .astype(str)
      .str.replace("\n", " ", regex=False)
)

df["form_of_violence_list"] = df["form_of_violence"].str.split(";")

df_long = df.explode("form_of_violence_list")

df_long["form_of_violence_list"] = (
    df_long["form_of_violence_list"]
      .astype(str)
      .str.strip()
)

df_long = df_long[df_long["form_of_violence_list"] != ""]


In [10]:
panel_total = (
    df.groupby(["country", "start_year"])
      .size()
      .reset_index(name="n_measures_total")
)


In [11]:
panel_by_violence = (
    df_long.groupby(["country", "start_year", "form_of_violence_list"])
           .size()
           .reset_index(name="n_measures")
)

panel_violence_wide = panel_by_violence.pivot_table(
    index=["country", "start_year"],
    columns="form_of_violence_list",
    values="n_measures",
    fill_value=0
).reset_index()


In [12]:
df["measure_family"] = (
    df["type_of_measure"]
      .astype(str)
      .str.split(">")
      .str[0]
      .str.strip()
)

panel_by_family = (
    df.groupby(["country", "start_year", "measure_family"])
      .size()
      .reset_index(name="n_measures")
)

panel_family_wide = panel_by_family.pivot_table(
    index=["country", "start_year"],
    columns="measure_family",
    values="n_measures",
    fill_value=0
).reset_index()


In [13]:
panel = panel_total.merge(
    panel_violence_wide,
    on=["country", "start_year"],
    how="left"
)

panel = panel.merge(
    panel_family_wide,
    on=["country", "start_year"],
    how="left"
)

panel = panel.fillna(0)
panel.head()


,country,start_year,n_measures_total,Child early and forced marriage,Domestic violence/Intimate partner violence,Female genital mutilation,Femicide,Femicide Feminicide,Forced pregnancy,Forced sterilization,...,Monitoring and Evaluation,Monitoring and evaluation,Perpetrators Programme,Policies,Prevention,Regional Initiatives,Regional/International Initiatives,Research and statistical data,Services,laws
0,Afghanistan,2003,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,Afghanistan,2004,5,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Afghanistan,2005,3,1,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,Afghanistan,2007,5,2,1,0,0,0,0,0,...,0,0,0,2,0,0,0,2,1,0
4,Afghanistan,2008,1,0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0


In [14]:
panel.columns = (
    panel.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("/", "_")
    .str.replace("-", "_")
)

panel.to_csv("unwomen_measures_country_year_panel_clean.csv", index=False)


In [16]:
import pandas as pd

path = "unwomen_measures_country_year_panel_clean.csv"
panel = pd.read_csv(path)

print("Shape:", panel.shape)
print("\nColumn count:", len(panel.columns))
print("\nDuplicate column names?:", panel.columns.duplicated().any())

print("\nFirst 5 rows:")
display(panel.head())

print("\nLast 5 rows:")
display(panel.tail())


Shape: (3130, 43)

Column count: 43

Duplicate column names?: False

First 5 rows:


,country,start_year,n_measures_total,child_early_and_forced_marriage,domestic_violence_intimate_partner_violence,female_genital_mutilation,femicide,femicide_feminicide,forced_pregnancy,forced_sterilization,...,monitoring_and_evaluation,monitoring_and_evaluation.1,perpetrators_programme,policies,prevention,regional_initiatives,regional_international_initiatives,research_and_statistical_data,services,laws.1
0,Afghanistan,2003,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,Afghanistan,2004,5,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Afghanistan,2005,3,1,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,Afghanistan,2007,5,2,1,0,0,0,0,0,...,0,0,0,2,0,0,0,2,1,0
4,Afghanistan,2008,1,0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0



Last 5 rows:


,country,start_year,n_measures_total,child_early_and_forced_marriage,domestic_violence_intimate_partner_violence,female_genital_mutilation,femicide,femicide_feminicide,forced_pregnancy,forced_sterilization,...,monitoring_and_evaluation,monitoring_and_evaluation.1,perpetrators_programme,policies,prevention,regional_initiatives,regional_international_initiatives,research_and_statistical_data,services,laws.1
3125,Zimbabwe,2017,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3126,Zimbabwe,2019,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3127,Zimbabwe,2021,5,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,2,0
3128,Zimbabwe,2022,5,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,2,0
3129,Zimbabwe,2023,3,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,2,0


In [17]:
import pandas as pd

panel = pd.read_csv("unwomen_measures_country_year_panel_clean.csv")

# Merge monitoring_and_evaluation
if "monitoring_and_evaluation.1" in panel.columns:
    panel["monitoring_and_evaluation"] = (
        panel["monitoring_and_evaluation"] +
        panel["monitoring_and_evaluation.1"]
    )
    panel = panel.drop(columns=["monitoring_and_evaluation.1"])

# Merge laws
if "laws.1" in panel.columns:
    panel["laws"] = (
        panel["laws"] +
        panel["laws.1"]
    )
    panel = panel.drop(columns=["laws.1"])


In [18]:
if "femicide_feminicide" in panel.columns:
    panel["femicide"] = (
        panel["femicide"] +
        panel["femicide_feminicide"]
    )
    panel = panel.drop(columns=["femicide_feminicide"])


In [19]:
print("New shape:", panel.shape)
print("Duplicate columns?:", panel.columns.duplicated().any())


New shape: (3130, 40)
Duplicate columns?: False


In [20]:
panel = panel.sort_values(["country","start_year"])

panel["institutional_depth"] = (
    panel.groupby("country")["n_measures_total"]
         .cumsum()
)


In [22]:
print("New shape:", panel.shape)
print("Duplicate columns?:", panel.columns.duplicated().any())
panel.head()

New shape: (3130, 41)
Duplicate columns?: False


,country,start_year,n_measures_total,child_early_and_forced_marriage,domestic_violence_intimate_partner_violence,female_genital_mutilation,femicide,forced_pregnancy,forced_sterilization,other_harmful_practices,...,laws,monitoring_and_evaluation,perpetrators_programme,policies,prevention,regional_initiatives,regional_international_initiatives,research_and_statistical_data,services,institutional_depth
0,Afghanistan,2003,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
1,Afghanistan,2004,5,0,0,0,0,0,0,0,...,4,0,0,0,0,0,0,0,0,6
2,Afghanistan,2005,3,1,0,0,0,0,0,0,...,2,0,0,1,0,0,0,0,0,9
3,Afghanistan,2007,5,2,1,0,0,0,0,0,...,0,0,0,2,0,0,0,2,1,14
4,Afghanistan,2008,1,0,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,15


In [23]:
# Export (clean + cumulative)
out_path = "unwomen_measures_country_year_panel_FINAL_with_depth.csv"

panel.to_csv(out_path, index=False, encoding="utf-8")
print("Saved:", out_path)


Saved: unwomen_measures_country_year_panel_FINAL_with_depth.csv


In [25]:
import pandas as pd

def read_csv_robust(path, sep=","):
    encodings = ["utf-8-sig", "utf-8", "cp1252", "latin1"]
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, sep=sep, encoding=enc)
        except UnicodeDecodeError as e:
            last_err = e
    raise last_err

# HDR
hdr_path = "external_country_datasets_v2/undp_hdr/HDR_composite_indices_complete_time_series.csv"
hdr = read_csv_robust(hdr_path)
print(hdr.shape)
print(hdr.columns)
print(hdr.head())

# V-Dem / OWID
elec_path = "external_country_datasets_v2/vdem_owid/electoral-democracy-index.csv"
vdem_owid_elec = read_csv_robust(elec_path)
print(vdem_owid_elec.head())

demo_path = "external_country_datasets_v2/vdem_owid/liberal-democracy-index.csv"
vdem_owid_demo = read_csv_robust(demo_path)
print(vdem_owid_demo.head())

wpe_path = "external_country_datasets_v2/vdem_owid/women-political-empowerment-index.csv"
vdem_owid_wpe = read_csv_robust(wpe_path)
print(vdem_owid_wpe.head())


(206, 1112)
Index(['iso3', 'country', 'hdicode', 'region', 'hdi_rank_2023', 'hdi_1990',
       'hdi_1991', 'hdi_1992', 'hdi_1993', 'hdi_1994',
       ...
       'pop_total_2014', 'pop_total_2015', 'pop_total_2016', 'pop_total_2017',
       'pop_total_2018', 'pop_total_2019', 'pop_total_2020', 'pop_total_2021',
       'pop_total_2022', 'pop_total_2023'],
      dtype='object', length=1112)
  iso3      country    hdicode region  hdi_rank_2023  hdi_1990  hdi_1991  \
0  AFG  Afghanistan        Low     SA          181.0     0.285     0.291   
1  ALB      Albania  Very High    ECA           71.0     0.654     0.638   
2  DZA      Algeria       High     AS           96.0     0.595     0.596   
3  AND      Andorra  Very High    NaN           32.0       NaN       NaN   
4  AGO       Angola     Medium    SSA          148.0       NaN       NaN   

   hdi_1992  hdi_1993  hdi_1994  ...  pop_total_2014  pop_total_2015  \
0     0.301     0.311     0.305  ...       32.792523       33.831764   
1     0.

In [26]:
import pandas as pd

# UN Women panel (your cleaned one with institutional_depth)
panel = pd.read_csv("unwomen_measures_country_year_panel_FINAL_with_depth.csv")

# Standardize key column name
panel = panel.rename(columns={"start_year": "year"})

# HDR (already loaded in your notebook as `hdr`)
# hdr has: iso3, country, ...
country_to_iso3 = hdr[["country", "iso3"]].dropna().drop_duplicates()

panel = panel.merge(country_to_iso3, on="country", how="left")

print("Missing ISO3:", panel["iso3"].isna().sum())
print(panel[panel["iso3"].isna()][["country"]].drop_duplicates().head(20))


Missing ISO3: 158
                                                country
837                                       Côte d’Ivoire
848                    Democratic Republic of the Congo
1017                                           Eswatini
2032                       Netherlands (Kingdom of the)
2337                                  Republic of Korea
2355                                Republic of Moldova
2682                                 State of Palestine
2874                                             Turkey
2960  United Kingdom of Great Britain and Northern I...
2985                        United Republic of Tanzania
3005                           United States of America


In [27]:
[c for c in hdr.columns if c.startswith("hdi_")][:5], [c for c in hdr.columns if c.startswith("gdi_")][:5], [c for c in hdr.columns if c.startswith("gii_")][:5]


(['hdi_rank_2023', 'hdi_1990', 'hdi_1991', 'hdi_1992', 'hdi_1993'],
 ['gdi_group_2023', 'gdi_1990', 'gdi_1991', 'gdi_1992', 'gdi_1993'],
 ['gii_rank_2023', 'gii_1990', 'gii_1991', 'gii_1992', 'gii_1993'])

In [28]:
import re

def hdr_wide_to_long(hdr, prefix):
    cols = ["iso3", "country"] + [c for c in hdr.columns if c.startswith(prefix + "_")]
    tmp = hdr[cols].copy()

    long = tmp.melt(
        id_vars=["iso3", "country"],
        var_name="var",
        value_name=prefix
    )

    long["year"] = long["var"].str.extract(r"_(\d{4})").astype("Int64")
    long = long.drop(columns=["var"])

    return long

hdr_hdi = hdr_wide_to_long(hdr, "hdi")
hdr_gdi = hdr_wide_to_long(hdr, "gdi")
hdr_gii = hdr_wide_to_long(hdr, "gii")

hdr_long = hdr_hdi.merge(hdr_gdi, on=["iso3","country","year"], how="outer")
hdr_long = hdr_long.merge(hdr_gii, on=["iso3","country","year"], how="outer")

print(hdr_long.shape)
print(hdr_long.head())


(23690, 6)
  iso3      country      hdi  year   gdi      gii
0  AFG  Afghanistan  181.000  2023  5.00  168.000
1  AFG  Afghanistan  181.000  2023  5.00    0.661
2  AFG  Afghanistan  181.000  2023  0.66  168.000
3  AFG  Afghanistan  181.000  2023  0.66    0.661
4  AFG  Afghanistan    0.496  2023  5.00  168.000


In [29]:
def owid_clean(df, value_col_name):
    df = df.copy()
    df = df.rename(columns={
        "Entity": "country_owid",
        "Code": "iso3",
        "Year": "year"
    })
    # Find the indicator column (the numeric one)
    indicator_cols = [c for c in df.columns if c not in ["country_owid","iso3","year","World region according to OWID"]]
    assert len(indicator_cols) == 1, indicator_cols
    df = df[["iso3","year", indicator_cols[0]]].rename(columns={indicator_cols[0]: value_col_name})
    return df

elec = owid_clean(vdem_owid_elec, "electoral_democracy")
ldem = owid_clean(vdem_owid_demo, "liberal_democracy")
wpe  = owid_clean(vdem_owid_wpe,  "women_political_empowerment")

vdem_all = elec.merge(ldem, on=["iso3","year"], how="outer")
vdem_all = vdem_all.merge(wpe,  on=["iso3","year"], how="outer")

print(vdem_all.head())


  iso3  year  electoral_democracy  liberal_democracy  \
0  AFG  1789                 0.02              0.029   
1  AFG  1790                 0.02              0.029   
2  AFG  1791                 0.02              0.029   
3  AFG  1792                 0.02              0.029   
4  AFG  1793                 0.02              0.029   

   women_political_empowerment  
0                        0.065  
1                        0.065  
2                        0.065  
3                        0.065  
4                        0.065  


In [31]:
# Check uniqueness in your panel (must be 1 row per iso3-year)
print("panel duplicates iso3-year:", panel.duplicated(["iso3","year"]).sum())

# Check HDR-long uniqueness
print("hdr_long duplicates iso3-year:", hdr_long.duplicated(["iso3","year"]).sum())

# Check V-Dem uniqueness
print("vdem_all duplicates iso3-year:", vdem_all.duplicated(["iso3","year"]).sum())


panel duplicates iso3-year: 125
hdr_long duplicates iso3-year: 16686
vdem_all duplicates iso3-year: 427500


In [32]:
hdr_long = hdr_long.sort_values(["iso3","year"])
hdr_long = hdr_long.groupby(["iso3","year"], as_index=False).agg({
    "country": "first",
    "hdi": "mean",
    "gdi": "mean",
    "gii": "mean"
})
print("hdr_long now duplicates:", hdr_long.duplicated(["iso3","year"]).sum())


hdr_long now duplicates: 0


In [33]:
# If your panel still has country column, keep it
panel = panel.copy()

master = panel.merge(
    hdr_long[["iso3","year","hdi","gdi","gii"]],
    on=["iso3","year"],
    how="left"
)

master = master.merge(
    vdem_all[["iso3","year","electoral_democracy","liberal_democracy","women_political_empowerment"]],
    on=["iso3","year"],
    how="left"
)

print("MASTER SHAPE:", master.shape)
print(master[["country","iso3","year","institutional_depth","hdi","gdi","gii",
              "electoral_democracy","liberal_democracy","women_political_empowerment"]].head())


MASTER SHAPE: (650198, 48)
       country iso3  year  institutional_depth      hdi   gdi   gii  \
0  Afghanistan  AFG  2003                    1  0.39200   NaN   NaN   
1  Afghanistan  AFG  2004                    6  0.40800   NaN   NaN   
2  Afghanistan  AFG  2005                    9  0.41700   NaN   NaN   
3  Afghanistan  AFG  2007                   14  0.44200   NaN   NaN   
4  Afghanistan  AFG  2008                   15  0.43792  0.69  0.69   

   electoral_democracy  liberal_democracy  women_political_empowerment  
0                0.227              0.086                        0.293  
1                0.238              0.091                        0.293  
2                0.319              0.113                        0.314  
3                0.373              0.200                        0.463  
4                0.379              0.202                        0.464  


In [34]:
master.to_csv("MASTER_unwomen_HDR_VDEM_panel.csv", index=False, encoding="utf-8")
print("Saved: MASTER_unwomen_HDR_VDEM_panel.csv")


Saved: MASTER_unwomen_HDR_VDEM_panel.csv


In [35]:
nan_counts = master[[
    "hdi","gdi","gii",
    "electoral_democracy",
    "liberal_democracy",
    "women_political_empowerment"
]].isna().sum()

print(nan_counts)


hdi                            647506
gdi                            647725
gii                            647873
electoral_democracy             81268
liberal_democracy               81269
women_political_empowerment     81281
dtype: int64


In [36]:
master.groupby("year")[["hdi","gdi","gii"]].apply(lambda x: x.isna().sum())


,hdi,gdi,gii
year,,,
1789,210,210,210
1838,1,1,1
1872,1,1,1
1900,1,1,1
1908,1,1,1
...,...,...,...
2021,20251,20253,20259
2022,30375,30377,30381
2023,162000,162001,162006


In [37]:
(master.shape[0] / panel.shape[0])


207.73099041533547

In [38]:
print("panel dup iso3-year:", panel.duplicated(["iso3","year"]).sum())
print("hdr_long dup iso3-year:", hdr_long.duplicated(["iso3","year"]).sum())
print("vdem_all dup iso3-year:", vdem_all.duplicated(["iso3","year"]).sum())


panel dup iso3-year: 125
hdr_long dup iso3-year: 0
vdem_all dup iso3-year: 427500


In [39]:
# Keep only needed columns first (safety)
vdem_all = vdem_all[["iso3","year","electoral_democracy","liberal_democracy","women_political_empowerment"]].copy()

# Collapse duplicates
vdem_all_u = (
    vdem_all.groupby(["iso3","year"], as_index=False)
            .agg({
                "electoral_democracy":"mean",
                "liberal_democracy":"mean",
                "women_political_empowerment":"mean"
            })
)

print("vdem_all_u rows:", vdem_all_u.shape[0])
print("vdem_all_u dup iso3-year:", vdem_all_u.duplicated(["iso3","year"]).sum())
print(vdem_all_u.head())


vdem_all_u rows: 31650
vdem_all_u dup iso3-year: 0
  iso3  year  electoral_democracy  liberal_democracy  \
0  ABW  2023                  NaN                NaN   
1  AFG  1789                 0.02              0.029   
2  AFG  1790                 0.02              0.029   
3  AFG  1791                 0.02              0.029   
4  AFG  1792                 0.02              0.029   

   women_political_empowerment  
0                          NaN  
1                        0.065  
2                        0.065  
3                        0.065  
4                        0.065  


In [40]:
dups = panel[panel.duplicated(["iso3","year"], keep=False)].sort_values(["iso3","year"])
print("Duplicated iso3-year rows:", len(dups))
display(dups[["country","iso3","year","n_measures_total","institutional_depth"]].head(30))


Duplicated iso3-year rows: 153


,country,iso3,year,n_measures_total,institutional_depth
1018,Eswatini,NaN,1991,1,2
2032,Netherlands (Kingdom of the),NaN,1991,1,1
837,Côte d’Ivoire,NaN,1998,1,1
2985,United Republic of Tanzania,NaN,1998,2,2
2338,Republic of Korea,NaN,1999,1,2
2961,United Kingdom of Great Britain and Northern I...,NaN,1999,1,2
2033,Netherlands (Kingdom of the),NaN,2000,1,2
2339,Republic of Korea,NaN,2000,2,4
2962,United Kingdom of Great Britain and Northern I...,NaN,2000,1,3
2963,United Kingdom of Great Britain and Northern I...,NaN,2001,1,4


In [41]:
# Identify which columns are measure counts (everything numeric except year)
count_cols = [c for c in panel.columns if c not in ["country","iso3","year"]]

panel_u = (
    panel.groupby(["iso3","year"], as_index=False)[count_cols].sum()
)

# Add country name back (first non-null)
country_map = panel[["iso3","country"]].dropna().drop_duplicates("iso3")
panel_u = panel_u.merge(country_map, on="iso3", how="left")

# Reorder columns nicely
panel_u = panel_u[["country","iso3","year"] + count_cols]

# Recompute institutional_depth from n_measures_total (after aggregation)
panel_u = panel_u.sort_values(["iso3","year"])
panel_u["institutional_depth"] = panel_u.groupby("iso3")["n_measures_total"].cumsum()

print("panel_u rows:", panel_u.shape[0])
print("panel_u dup iso3-year:", panel_u.duplicated(["iso3","year"]).sum())
panel_u.head()


panel_u rows: 2972
panel_u dup iso3-year: 0


,country,iso3,year,n_measures_total,child_early_and_forced_marriage,domestic_violence_intimate_partner_violence,female_genital_mutilation,femicide,forced_pregnancy,forced_sterilization,...,laws,monitoring_and_evaluation,perpetrators_programme,policies,prevention,regional_initiatives,regional_international_initiatives,research_and_statistical_data,services,institutional_depth
0,Afghanistan,AFG,2003,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
1,Afghanistan,AFG,2004,5,0,0,0,0,0,0,...,4,0,0,0,0,0,0,0,0,6
2,Afghanistan,AFG,2005,3,1,0,0,0,0,0,...,2,0,0,1,0,0,0,0,0,9
3,Afghanistan,AFG,2007,5,2,1,0,0,0,0,...,0,0,0,2,0,0,0,2,1,14
4,Afghanistan,AFG,2008,1,0,1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,15


In [42]:
master_u = (
    panel_u.merge(hdr_long, on=["iso3","year"], how="left")
           .merge(vdem_all_u, on=["iso3","year"], how="left")
)

print("MASTER_U SHAPE:", master_u.shape)
print("MASTER_U dup iso3-year:", master_u.duplicated(["iso3","year"]).sum())

master_u[["country","iso3","year","institutional_depth","hdi","gdi","gii",
          "electoral_democracy","liberal_democracy","women_political_empowerment"]].head()


MASTER_U SHAPE: (2972, 49)
MASTER_U dup iso3-year: 0


KeyError: "['country'] not in index"

In [43]:
 # If country columns exist under different names, pick one
country_cols = [c for c in master_u.columns if "country" in c.lower()]

print("Country-like columns:", country_cols)

if "country" not in master_u.columns:
    if "country_x" in master_u.columns:
        master_u = master_u.rename(columns={"country_x": "country"})
    elif "country_y" in master_u.columns:
        master_u = master_u.rename(columns={"country_y": "country"})
    elif len(country_cols) > 0:
        master_u = master_u.rename(columns={country_cols[0]: "country"})
    else:
        # fallback: rebuild country from panel_u mapping
        country_map = panel_u[["iso3","country"]].drop_duplicates("iso3")
        master_u = master_u.merge(country_map, on="iso3", how="left")

# If there are still leftover country_* columns, drop them
drop_cols = [c for c in master_u.columns if c.lower().startswith("country_")]
master_u = master_u.drop(columns=drop_cols, errors="ignore")

print("Now has country?:", "country" in master_u.columns)
print(master_u[["country","iso3","year"]].head())


Country-like columns: ['country_x', 'country_y']
Now has country?: True
       country iso3  year
0  Afghanistan  AFG  2003
1  Afghanistan  AFG  2004
2  Afghanistan  AFG  2005
3  Afghanistan  AFG  2007
4  Afghanistan  AFG  2008


In [44]:
master_u[["country","iso3","year","institutional_depth","hdi","gdi","gii",
          "electoral_democracy","liberal_democracy","women_political_empowerment"]].head()


,country,iso3,year,institutional_depth,hdi,gdi,gii,electoral_democracy,liberal_democracy,women_political_empowerment
0,Afghanistan,AFG,2003,1,0.39200,NaN,NaN,0.227,0.086,0.293
1,Afghanistan,AFG,2004,6,0.40800,NaN,NaN,0.238,0.091,0.293
2,Afghanistan,AFG,2005,9,0.41700,NaN,NaN,0.319,0.113,0.314
3,Afghanistan,AFG,2007,14,0.44200,NaN,NaN,0.373,0.200,0.463
4,Afghanistan,AFG,2008,15,0.43792,0.69,0.69,0.379,0.202,0.464


In [45]:
master_u.to_csv("MASTER_unwomen_HDR_VDEM_panel_CLEAN.csv", index=False, encoding="utf-8")
print("Saved: MASTER_unwomen_HDR_VDEM_panel_CLEAN.csv")


Saved: MASTER_unwomen_HDR_VDEM_panel_CLEAN.csv


In [46]:
master_u

,country,iso3,year,n_measures_total,child_early_and_forced_marriage,domestic_violence_intimate_partner_violence,female_genital_mutilation,femicide,forced_pregnancy,forced_sterilization,...,regional_international_initiatives,research_and_statistical_data,services,institutional_depth,hdi,gdi,gii,electoral_democracy,liberal_democracy,women_political_empowerment
0,Afghanistan,AFG,2003,1,0,0,0,0,0,0,...,0,1,0,1,0.392000,NaN,NaN,0.227,0.086,0.293
1,Afghanistan,AFG,2004,5,0,0,0,0,0,0,...,0,0,0,6,0.408000,NaN,NaN,0.238,0.091,0.293
2,Afghanistan,AFG,2005,3,1,0,0,0,0,0,...,0,0,0,9,0.417000,NaN,NaN,0.319,0.113,0.314
3,Afghanistan,AFG,2007,5,2,1,0,0,0,0,...,0,2,1,14,0.442000,NaN,NaN,0.373,0.200,0.463
4,Afghanistan,AFG,2008,1,0,1,0,0,0,0,...,0,0,0,15,0.437920,0.690,0.6900,0.379,0.202,0.464
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2967,Zimbabwe,ZWE,2017,1,0,0,0,0,0,0,...,0,0,1,27,0.574970,0.937,0.5200,0.295,0.221,0.790
2968,Zimbabwe,ZWE,2019,1,0,0,0,0,0,0,...,0,1,0,28,0.584159,0.929,0.5250,0.292,0.197,0.788
2969,Zimbabwe,ZWE,2021,5,0,0,0,0,0,0,...,0,0,2,33,0.580996,0.955,0.5190,0.289,0.188,0.791
2970,Zimbabwe,ZWE,2022,5,1,0,0,0,0,0,...,0,1,2,38,0.593745,0.938,0.5200,0.286,0.180,0.775
